In [1]:
#Importamos todas las librerias que necesitamos para trabajar con mysqlconnector

import time
import requests
from sqlalchemy import create_engine, MetaData, Table, update, select, or_
import pandas as pd

In [ ]:
usuario = "root"
password = "xxxxxxx"
host = "127.0.0.1"
base_de_datos = "pandemusic" # Asegúrate de haber creado el Schema en Workbench

In [3]:
# Creamos el engine para mysql alchemy

engine = create_engine(f"mysql+mysqlconnector://{usuario}:{password}@{host}/{base_de_datos}")
metadata = MetaData()

In [4]:
# Esto hace que SQLAlchemy "aprenda" como son las columnas de la tabla
tabla_artistas = Table('tabla_artistas_limpia', metadata, autoload_with=engine)

In [5]:
api_key_last_fm = "dffc96fb5e37b49d6382379a54a66094"
headers = {'User-Agent': 'MiBootcampApp/1.0'} # <--- Esto es vital para evitar el 403
url = "http://ws.audioscrobbler.com/2.0/"


In [ ]:
#Ahora usamos el engine con mysql Alchemy para poder saber cuantos nulos hay que buscar, queda almacenado en artistas_pendientes

with engine.connect() as conn:
    query = select(tabla_artistas.c.artista).where(
        or_(tabla_artistas.c.biografia == None, tabla_artistas.c.biografia == "")
    )
    artistas_pendientes = conn.execute(query).fetchall()

print(f"Total de artistas por actualizar: {len(artistas_pendientes)}")        


Total de artistas por actualizar: 2018


In [10]:
# Iteramos sobre la lista de pendientes 
for fila in artistas_pendientes:
    artista = fila[0]
    # Reutilizamos el codigo de llamada a la API
    parametros = {'method': 'artist.getinfo', 'artist': artista, 'api_key': api_key_last_fm, 'format': 'json'}
    headers = {'User-Agent': 'MiAppRescate/1.0'}
    
    try:
        response = requests.get("http://ws.audioscrobbler.com/2.0/", params=parametros, headers=headers)
        
        if response.status_code == 200: #si la conexion es exitosa, transformar en json
            datos_api = response.json()
            
            if 'artist' in datos_api:
                info = datos_api['artist']
                
                # Extraemos los datos como estan en la tabla
                stats = info.get('stats', {})
                audiencia = stats.get('listeners', 0)
                reproducciones = stats.get('playcount', 0)
                lista_similares = [s['name'] for s in info.get('similar', {}).get('artist', [])]
                primer_similar = lista_similares[0] if lista_similares else "Sin similares" #corrijo el tema de las listas
                bio = info.get('bio', {}).get('content', 'Sin biografia')
                
                # ACTUALIZAMOS LA BASE DE DATOS en SQL
                with engine.begin() as conn_update:
                    stmt = (
                        update(tabla_artistas)
                        .where(tabla_artistas.c.artista == artista)
                        .values(
                            audiencia=audiencia,
                            reproducciones=reproducciones,
                            similares=primer_similar,  # Aquí guardamos solo el primero
                            biografia=bio
                        )
                    )
                    conn_update.execute(stmt)
                
            else:
                # Marcamos como no encontrado para no volver a intentarlo
                with engine.begin() as conn_update:
                    conn_update.execute(update(tabla_artistas).where(tabla_artistas.c.artista == artista).values(biografia="No encontrado en API"))
                print(f"⚠️ {artista} no existe en Last.fm.")
        
        elif response.status_code == 403:
            print("❌ Error 403: Revisa tu API Key.")
            break
            
        time.sleep(0.2) # Respiro para la API
        
    except Exception as e:
        print(f"💥 Error inesperado con {artista}: {e}")

⚠️ 24 Kara no existe en Last.fm.
⚠️ 31st Youngin' no existe en Last.fm.
⚠️ 3rdworld Boski no existe en Last.fm.
⚠️ 7.0.4 no existe en Last.fm.
⚠️ Ada Lattanzi & Calo Divinti no existe en Last.fm.
⚠️ Ágúst Karel no existe en Last.fm.
⚠️ Alexis Velez no existe en Last.fm.
⚠️ Amirah Tiye no existe en Last.fm.
⚠️ Anthony Stephens Jr. no existe en Last.fm.
⚠️ Antón Dávila no existe en Last.fm.
⚠️ Antonio Fernández "El Farru" no existe en Last.fm.
⚠️ Arqueezy no existe en Last.fm.
⚠️ Asidity7 no existe en Last.fm.
⚠️ Àtomo no existe en Last.fm.
⚠️ Awkward African no existe en Last.fm.
⚠️ B-Staz no existe en Last.fm.
⚠️ B. Rog no existe en Last.fm.
⚠️ Baby Honcho no existe en Last.fm.
⚠️ Bassirou Koureissi no existe en Last.fm.
⚠️ Bell Zed no existe en Last.fm.
⚠️ Benboy no existe en Last.fm.
⚠️ Benny Blanco 650 no existe en Last.fm.
⚠️ BlackMago no existe en Last.fm.
⚠️ BlastTeam Swirl no existe en Last.fm.
⚠️ Blvck Frost no existe en Last.fm.
⚠️ Brayzee Loww no existe en Last.fm.
⚠️ Brayzee